In [1]:
from lsst_efd_client import EfdClient 
import numpy as np
from lsst.summit.utils import ConsDbClient

URL = "http://consdb-pq.consdb:8080/consdb"  
os.environ["no_proxy"] += ",.consdb"
cdb_client = ConsDbClient(URL)
    
    
visit=None
query_table = f"""
        SELECT
            visit_id AS visit_id,
            can_see_sky AS sky
        FROM
            cdb_lsstcam.visit1

""" 


query_data = f"""
        SELECT
            exposure_id AS id,
            altitude AS elevation,
            azimuth AS azimuth
        FROM
            cdb_lsstcam.exposure AS e

""" 


In [13]:
import pandas as pd
from shapely import contains

result_df = cdb_client.query(query_table).to_pandas()

see_sky=result_df
# print(see_sky)
visitsl=[]
search_art= "tape"     #['tape','horseshoe','banana','airplane','pillow','moon','nibble','ghoul','arc','smudge']
df = pd.read_csv("stray_light_20260506.csv")

df = df[~df['img_type'].str.extract(r'(dark|flat|bias)')]



df['exposure_id'] = df['exposure_id'].astype(int)
result_df['visit_id']   = result_df['visit_id'].astype(int)

# keep only result rows that are present in df
result_df = result_df[result_df['visit_id'].isin(df['exposure_id'])].copy()

# build mapping visit_id -> can_see_sky (coerce to bool)
visit_mask = result_df.drop_duplicates('visit_id').set_index('visit_id')['sky'].astype(bool)

# map mask onto df (missing visits -> False)
df['sky'] = df['exposure_id'].map(visit_mask).fillna(False)

# now you can filter or use the mask
df_visible    = df[df['sky']]
df_not_visible = df[~df['sky']]

print(f"Visits in df: {len(df)}; matched in result_df: {len(result_df)}")
print(f"Visible (can_see_sky=True): {len(df_visible)}; Not visible: {len(df_not_visible)}")
print(df_visible.shape)


# remove exposures that are dark/flat/bias img types
orig_len = len(df)
contaminated = df_visible[df_visible['flag'] > 0]
print(f"Filtered img_type dark/flat/bias: {orig_len} -> {len(df_visible)}")

# print(df_stray)
print("Number of visits with stray light artifacts:", len(contaminated), "/", len(df_visible),"----", len(contaminated) / len(df_visible) * 100, "%")

for art in ['tape','horseshoe','banana','airplane','pillow','moon','nibble','ghoul','arc','smudge']:
    mask = df_visible['artifact'].str.contains(art, case=False, na=False)
    df_stray = df_visible[mask]
    if art==search_art:
        df_selected=df_stray
    print(f"Visits with {art}: {len(df_stray)} / {len(df_visible)} ----- {len(df_stray) / len(df_visible) * 100:.3f} %, of the strayligh it contributes {len(df_stray) / len(contaminated) * 100:.3f} %" )
    #PRINT A SUBSET OF VISITS AFFECTEED BY EACH KNOWN ARTIFACT
    # if len(df_stray) > 0:
    #     sample_size = min(10, len(df_stray))
    #     np.random.seed(42)
    #     print(np.random.choice(df_stray['exposure_id'].values.astype(int), size=sample_size, replace=False))


TypeError: bad operand type for unary ~: 'float'

In [9]:
print(df_selected)

         exposure_id         exposure_name   day_obs  seq_num physical_filter  \
163    2025041700633  MC_O_20250417_000633  20250417      633            i_39   
164    2025041700634  MC_O_20250417_000634  20250417      634            i_39   
165    2025041700635  MC_O_20250417_000635  20250417      635            i_39   
171    2025041700641  MC_O_20250417_000641  20250417      641            i_39   
172    2025041700642  MC_O_20250417_000642  20250417      642            i_39   
...              ...                   ...       ...      ...             ...   
82893  2026030300603  MC_O_20260303_000603  20260303      603            y_10   
82940  2026030300650  MC_O_20260303_000650  20260303      650            y_10   
82944  2026030300654  MC_O_20260303_000654  20260303      654            y_10   
82956  2026030300666  MC_O_20260303_000666  20260303      666            y_10   
82976  2026030300686  MC_O_20260303_000686  20260303      686            y_10   

      band          ra     